In [19]:
import json
import os
import random
import argparse
from typing import Any, Dict, List, Union

In [ ]:
def round_prob(p: float) -> float:
    try:
        f = float(p)
    except Exception:
        f = 0.0
    f = max(0.0, min(1.0, f))
    return round(f, 2)

In [ ]:
def bool_to_prob(
    val: Any,
    low_false: float = 0.0,
    high_false: float = 0.2,
    low_true: float = 0.8,
    high_true: float = 1.0
) -> float:
    if isinstance(val, bool):
        sampled = random.uniform(low_true, high_true) if val else random.uniform(low_false, high_false)
        return round_prob(sampled)
    try:
        f = float(val)
        return round_prob(f)
    except Exception:
        sampled = random.uniform(low_false, high_false)
        return round_prob(sampled)

In [16]:
def normalize_species_list(sp: Any) -> List[Dict[str, Any]]:
    out = []
    if not sp:
        return out
    if isinstance(sp, list):
        for item in sp:
            if isinstance(item, dict):
                name = item.get('species') or item.get('name') or item.get('label') or "unknown"
                prob = item.get('probability', item.get('prob', 1.0))
                out.append({"species": name, "probability": round_prob(prob)})
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                out.append({"species": item[0], "probability": round_prob(item[1])})
            elif isinstance(item, str):
                out.append({"species": item, "probability": round_prob(1.0)})
    elif isinstance(sp, dict):
        name = sp.get('species') or sp.get('name') or "unknown"
        prob = sp.get('probability', 1.0)
        out.append({"species": name, "probability": round_prob(prob)})
    elif isinstance(sp, str):
        out.append({"species": sp, "probability": round_prob(1.0)})
    return out

In [17]:
def normalize_diseases(dv: Any) -> List[Dict[str, Any]]:
    out = []
    if not dv:
        return out
    if isinstance(dv, list):
        for d in dv:
            if isinstance(d, dict):
                name = d.get('disease') or d.get('name') or "unknown"
                prob = d.get('probability', d.get('prob', random.uniform(0.8, 1.0)))
                out.append({"disease": name, "probability": round_prob(prob)})
            elif isinstance(d, str):
                out.append({"disease": d, "probability": round_prob(random.uniform(0.8, 1.0))})
    elif isinstance(dv, dict):
        name = dv.get('disease') or dv.get('name') or "unknown"
        prob = dv.get('probability', random.uniform(0.8, 1.0))
        out.append({"disease": name, "probability": round_prob(prob)})
    elif isinstance(dv, bool):
        if dv:
            out.append({"disease": "unknown", "probability": round_prob(random.uniform(0.8, 1.0))})
    elif isinstance(dv, str):
        out.append({"disease": dv, "probability": round_prob(random.uniform(0.8, 1.0))})
    return out

In [18]:
def normalize_annotation(ann: Dict[str, Any]) -> Dict[str, Any]:
    out: Dict[str, Any] = {}

    out['bbox'] = ann.get('bbox') or ann.get('box') or []

    plant_class = ann.get('plant_class') or ann.get('class') or ann.get('plantType') or ann.get('plant_classification')
    if isinstance(plant_class, str):
        if 'дерев' in plant_class.lower() or 'tree' in plant_class.lower():
            out['class'] = 'a tree'
        else:
            out['class'] = plant_class
    else:
        out['class'] = 'a tree'

    sp = ann.get('species_probabilities') or ann.get('species') or ann.get('species_probs') or ann.get('species_prob')
    out['species_probabilities'] = normalize_species_list(sp)

    hollows_val = ann.get('hollows') or ann.get('hollow') or ann.get('has_hollows')
    cracks_val = ann.get('cracks') or ann.get('crack') or ann.get('has_cracks')
    stem_damage_val = ann.get('stem_damage') or ann.get('stemDamage') or ann.get('stem_rot') or ann.get('stemRot')
    crown_damage_val = ann.get('crown_damage') or ann.get('crownDamage')
    fruiting_bodies_val = ann.get('fruiting_bodies') or ann.get('fruitingBodies') or ann.get('has_fruiting_bodies')

    out['hollows'] = bool_to_prob(hollows_val) if hollows_val is not None else 0.0
    out['cracks'] = bool_to_prob(cracks_val) if cracks_val is not None else 0.0
    out['stem_damage'] = bool_to_prob(stem_damage_val) if stem_damage_val is not None else 0.0
    out['crown_damage'] = bool_to_prob(crown_damage_val) if crown_damage_val is not None else 0.0
    out['fruiting_bodies'] = bool_to_prob(fruiting_bodies_val) if fruiting_bodies_val is not None else 0.0

    diseases_val = ann.get('diseases') or ann.get('diseases_probabilities') or ann.get('diseases_probs') or ann.get('disease') or ann.get('disease_probabilities')
    out['diseases'] = normalize_diseases(diseases_val)

    dbp = ann.get('dry_branches_percentage') or ann.get('dryBranches') or ann.get('dry_branches') or ann.get('dry')
    try:
        out['dry_branches_percentage'] = int(dbp) if dbp is not None else 0
    except Exception:
        out['dry_branches_percentage'] = 0

    return out

In [20]:
def process_input_file(path: str, out_dir: str, seed: Union[int, None] = None) -> List[str]:
    if seed is not None:
        random.seed(seed)
    os.makedirs(out_dir, exist_ok=True)
    created = []

    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if isinstance(data, list):
        for item in data:
            if not isinstance(item, dict):
                continue
            img_ref = item.get('img_path') or item.get('img_url') or item.get('img') or item.get('image')
            img_basename = os.path.basename(img_ref) if img_ref else None

            anns = item.get('annotations')
            if isinstance(anns, list):
                out_obj = {"img_path": img_basename or f"image_{len(created)+1}.jpg", "annotations": []}
                for ann in anns:
                    out_obj['annotations'].append(normalize_annotation(ann))
            elif isinstance(anns, dict):
                out_obj = {"img_path": img_basename or f"image_{len(created)+1}.jpg", "annotations": [normalize_annotation(anns)]}
            else:
                if 'img_path' in item and isinstance(item.get('annotations'), list):
                    out_obj = {"img_path": item['img_path'], "annotations": []}
                    for ann in item.get('annotations', []):
                        out_obj['annotations'].append(normalize_annotation(ann))
                else:
                    out_obj = {"img_path": img_basename or f"image_{len(created)+1}.jpg", "annotations": []}

            name_wo_ext = os.path.splitext(out_obj['img_path'])[0]
            out_path = os.path.join(out_dir, f"{name_wo_ext}.json")
            with open(out_path, 'w', encoding='utf-8') as of:
                json.dump(out_obj, of, ensure_ascii=False, indent=2)
            created.append(out_path)

    elif isinstance(data, dict):
        img_ref = data.get('img_path') or data.get('img_url') or data.get('img') or data.get('image')
        img_basename = os.path.basename(img_ref) if img_ref else "image_1.jpg"
        anns = data.get('annotations')
        if isinstance(anns, list):
            out_obj = {"img_path": img_basename, "annotations": [normalize_annotation(a) for a in anns]}
        elif isinstance(anns, dict):
            out_obj = {"img_path": img_basename, "annotations": [normalize_annotation(anns)]}
        else:
            out_obj = {"img_path": img_basename, "annotations": []}
        name_wo_ext = os.path.splitext(out_obj['img_path'])[0]
        out_path = os.path.join(out_dir, f"{name_wo_ext}.json")
        with open(out_path, 'w', encoding='utf-8') as of:
            json.dump(out_obj, of, ensure_ascii=False, indent=2)
        created.append(out_path)
    else:
        raise ValueError("Неизвестная структура входного JSON.")

    print(f"Создано {len(created)} файлов в {out_dir} из {path}")
    return created

In [21]:
def batch_process(input_path_or_dir: str, out_dir: str, seed: Union[int, None] = None) -> None:
    files = []
    if os.path.isdir(input_path_or_dir):
        for fname in os.listdir(input_path_or_dir):
            if fname.lower().endswith('.json'):
                files.append(os.path.join(input_path_or_dir, fname))
    elif os.path.isfile(input_path_or_dir):
        files = [input_path_or_dir]
    else:
        raise FileNotFoundError(input_path_or_dir)

    all_created = []
    for fp in files:
        created = process_input_file(fp, out_dir, seed=seed)
        all_created.extend(created)

    print(f"В итоге создано {len(all_created)} файлов.")

In [23]:
seed = 42
input = "data/gemini-2.5-pro, первые 45 штук.json"
out = "data/converted_jsons_new"

batch_process(input, out, seed=seed)

Создано 44 файлов в data/converted_jsons_new из data/gemini-2.5-pro, первые 45 штук.json
В итоге создано 44 файлов.


In [24]:
seed = 42
input = "data/Для кропнутых растений разметка от Прошки.json"
out = "data/converted_jsons_crop"

batch_process(input, out, seed=seed)

Создано 48 файлов в data/converted_jsons_crop из data/Для кропнутых растений разметка от Прошки.json
В итоге создано 48 файлов.
